# C: заданный угол, фиксированная сеть и отдельные токи

Шаг интегрирования **0.1 мс**, полные **Afferents**, без усиления момента и без ПД.
Конфигурация содержит **52 поля: 50 параметров контроллера + фиксированные a1 и a2**.
Описания всех параметров приведены на английском в `genome_cfg.json`.

По умолчанию загружается `results/best_genome.json`, если он существует; иначе показан **исходный необученный контроллер**. Для другого решения задайте `GENOME_PATH`: путь к `best_genome.json`, `final_population.npz` или `generations/generation_XXXX.npz`. При загрузке сохраняется геометрия исходного запуска.
Обучение и sweep не запускаются автоматически.

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

REPO = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src/SpikingNN/core/Izh_net.py").is_file())
sys.path.insert(0, str(REPO))
from scripts.AfferentBalance2026.experiments.core.genome import GenomeConfig
from scripts.AfferentBalance2026.experiments.simulation import case_grid, simulate
from scripts.AfferentBalance2026.experiments.visualization import (
    configure_fonts, load_solution, replay_grid, parameter_scan,
    plot_phase, plot_angles, plot_states,
)

EXPERIMENT = REPO / "scripts/AfferentBalance2026/experiments/C_setpoint_balance"
config = GenomeConfig(EXPERIMENT / "genome_cfg.json")
DEFAULT_GENOME = EXPERIMENT / "results/best_genome.json"
GENOME_PATH = DEFAULT_GENOME if DEFAULT_GENOME.is_file() else None
# Set None for the untrained configuration, or choose any saved population/generation.
RANK = 1
if GENOME_PATH is None:
    simulation_config, physical = config, config.initial_physical.copy()
    selection = {"selection": "initial controller (untrained)"}
else:
    simulation_config, physical, selection = load_solution(GENOME_PATH, rank=RANK)

# This environment variable is used only by automated notebook smoke checks.
SMOKE_TEST = os.environ.get("SPIKINGNN_NOTEBOOK_SMOKE") == "1"
DURATION_S = 0.02 if SMOKE_TEST else config.data["simulation"]["duration_s"]
TARGET_DEG = 0.0
CURRENTS = np.array([0.0, 0.0])  # External currents; not genome fields.
configure_fonts(13)
print(selection)
print("Fixed a1/a2 (m):", physical[46:48], "integration step: 0.1 ms")


In [ ]:
from scripts.AfferentBalance2026.experiments.C_setpoint_balance.polynomial_study import (
    load_current_law, polynomial_currents,
)
CURRENT_LAW_PATH = None  # Optional supplied current_law.json; no fitting here.
coefficients = None if CURRENT_LAW_PATH is None else load_current_law(CURRENT_LAW_PATH)
if coefficients is not None:
    CURRENTS = polynomial_currents(coefficients, np.deg2rad(TARGET_DEG))
print("Target (deg):", TARGET_DEG, "external currents:", CURRENTS)


## Функции для работы с выбранным геномом

`run_on_grid` рассчитывает сетку начальных условий; `view_states` показывает все основные переменные для одного примера. `scan_parameter` меняет один свободный параметр в допустимых пределах. Геометрия фиксирована: для её изменения создайте отдельный конфиг.

Фазовые графики сохраняют точки через 1 мс, но интегрирование и метрики всегда используют шаг 0.1 мс. Для подробного просмотра одной траектории записывается каждый шаг.

In [ ]:
def run_on_grid(angles_deg, speeds, target_deg=TARGET_DEG, currents=None):
    """Replay the selected genome on an absolute-angle initial-condition grid."""
    drive = CURRENTS if currents is None else currents
    return replay_grid(simulation_config, physical, angles_deg, speeds,
                       target_deg=target_deg, currents=drive,
                       duration_s=DURATION_S, stride=10)


def view_states(theta0_deg=30.0, omega0=0.0):
    """Plot limb, neurons, muscles and full afferents for one initial condition."""
    record = replay_grid(simulation_config, physical, [theta0_deg], [omega0],
                         target_deg=TARGET_DEG, currents=CURRENTS,
                         duration_s=DURATION_S, stride=1)[0]
    for name, figure in plot_states(record).items():
        print(name)
        display(figure)
        plt.close(figure)
    return record


def phase_portrait(records):
    """Display genuine phase trajectories, including failed recovery attempts."""
    figure = plot_phase(records)
    display(figure)
    plt.close(figure)


def scan_parameter(name, values):
    """Compare one mutable genome parameter without changing the saved genome."""
    records = parameter_scan(simulation_config, physical, name, values,
                             target_deg=TARGET_DEG, currents=CURRENTS,
                             duration_s=DURATION_S)
    figure = plot_angles(records)
    display(figure)
    plt.close(figure)
    return records


## Фазовый портрет: сетка углов и ненулевых скоростей

По умолчанию 10×10: углы −50…50°, скорости −1…1 рад/с. Звезда обозначает цель, крестики — падения. Неуспешные траектории не исключаются.

In [ ]:
angles = np.linspace(-50, 50, 2 if SMOKE_TEST else 10)
speeds = np.linspace(-1, 1, 2 if SMOKE_TEST else 10)
phase_records = run_on_grid(angles, speeds)
phase_portrait(phase_records)
print("Successful cases:", sum(r["metrics"][5] for r in phase_records), "/", len(phase_records))


## Временные ряды угла при нулевой скорости

In [ ]:
angle_records = run_on_grid(angles, [0.0])
figure = plot_angles(angle_records)
display(figure)
plt.close(figure)


## Состояния одного примера

Напряжения и спайки нейронов, активации/силы/кальций мышц, Ia/II/Ib и движение конечности. Все сигналы относятся только к выбранному начальному условию.

In [ ]:
single_record = view_states(theta0_deg=-30.0, omega0=0.0)

## Сетка значений одного параметра генома

In [ ]:
field = simulation_config.data["parameters"][16]  # Flexor neuromuscular weight.
weights = np.geomspace(field["minimum"], field["maximum"], 3 if SMOKE_TEST else 5)
parameter_records = scan_parameter(field["name"], weights)


## Пять заданных углов

Этот блок только применяет заданные токи или загруженный полином. Он **не обучает** сеть и не подбирает токи. При нулевых токах это проверка исходного контроллера на разные цели, а не готовое решение удержания угла.

In [ ]:
PLOT_ALL_TARGETS = False
if PLOT_ALL_TARGETS:
    for target in config.data["setpoint"]["targets_deg"]:
        drive = CURRENTS if coefficients is None else polynomial_currents(coefficients, np.deg2rad(target))
        records = run_on_grid(angles, speeds, target_deg=target, currents=drive)
        print("Target:", target, "currents:", drive)
        phase_portrait(records)
